# DroneAId — YOLOv12s Baseline Training on SARD (Kaggle GPU)

**Purpose:** Baseline training run #1 — YOLOv12s on SARD for human detection, without heavy augmentation or SAHI. This establishes the reference mAP against which all subsequent experiments (Albumentations aug, SAHI tile training, class balancing, etc.) are compared.

**Runtime requirements:**
- Accelerator: **GPU T4 x1** (or P100)
- Internet: **On** (to download pretrained YOLOv12s weights)
- Dataset: SARD attached as Kaggle input (Roboflow YOLO format)

**Reference EDA findings (from local integrity check):**
- 5,755 images total · 7,424 bounding boxes
- Split: train 4,041 / valid 1,144 / test 570
- **83.50% of bboxes are tiny** (normalized area < 0.02)
- Median bbox size: ~0.056 × 0.128 (w × h) relative to frame
- Drives `imgsz=1280` training decision

**Constraint targets:**
- C-A1: final `.onnx` size ≤ 50 MB (budget ~25 MB for this model, leave ~25 MB for seg model)
- C-A3: not validated here — validated separately on CPU notebook

## 1. Environment check

In [ ]:
!pip install -q ultralytics==8.4.37 onnx onnxruntime
import torch, platform, os, json
from pathlib import Path

print('torch        :', torch.__version__)
print('cuda avail   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu name     :', torch.cuda.get_device_name(0))
    print('gpu memory   :', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), 'GB')
print('python       :', platform.python_version())
print('platform     :', platform.machine())

assert torch.cuda.is_available(), 'GPU required for training — set Accelerator to GPU T4 x1'

## 2. Dataset path — UPDATE TO YOUR ATTACHED SARD

The SARD dataset must be attached via the Kaggle sidebar (Add data). The Roboflow SARD export ships with a `data.yaml` file that references `../train/images`, etc. We override with absolute paths for robustness.

In [ ]:
# TODO: set to the path where SARD was attached
SARD_ROOT = Path('/kaggle/input/search-and-rescue')  # change if dataset name differs

assert SARD_ROOT.exists(), f'SARD not found at {SARD_ROOT}. Attach the dataset first.'
for split in ('train', 'valid', 'test'):
    assert (SARD_ROOT / split / 'images').exists(), f'missing {split}/images'
    assert (SARD_ROOT / split / 'labels').exists(), f'missing {split}/labels'
print('SARD structure OK')

# Write a clean data.yaml under /kaggle/working (writable)
WORK = Path('/kaggle/working')
data_yaml = WORK / 'sard.yaml'
data_yaml.write_text(f'''path: {SARD_ROOT}
train: train/images
val: valid/images
test: test/images
nc: 1
names: ['human']
''')
print('wrote:', data_yaml)
print(data_yaml.read_text())

## 3. Training configuration

Baseline philosophy — minimal tweaks, default augmentation. The goal is a clean mAP baseline. Augmentation experiments (Albumentations heavy aug, SAHI tile training, class balancing for prone bodies) will be separate runs in later notebooks so we can attribute improvements.

| Key | Value | Reason |
|---|---|---|
| model | yolo12s.pt | 9.28M params, ~18 MB — fits dual-model 50 MB budget |
| imgsz | 1280 | EDA shows 83.5% tiny objects, median 0.056x0.128 |
| epochs | 100 | Enough for convergence, early stop handles overfit |
| patience | 30 | Early stop if val mAP plateaus |
| batch | -1 | Auto-batch (T4 has 15 GB, should fit imgsz=1280) |
| optimizer | auto | Ultralytics SGD→AdamW auto-switch |
| cos_lr | True | Smoother convergence at long schedule |
| amp | True | T4 supports fp16, ~1.5x training speedup |
| cache | disk | Kaggle has limited RAM; disk cache is safe |
| workers | 4 | Kaggle GPU notebook typically has 4 vCPU |

In [ ]:
from ultralytics import YOLO

RUN_NAME = 'yolov12s_sard_baseline_v1'
PROJECT = str(WORK / 'runs')

model = YOLO('yolo12s.pt')

results = model.train(
    data=str(data_yaml),
    imgsz=1280,
    epochs=100,
    patience=30,
    batch=-1,
    optimizer='auto',
    cos_lr=True,
    amp=True,
    cache='disk',
    workers=4,
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    seed=42,
    verbose=True,
    save=True,
    save_period=-1,
    plots=True,
)

## 4. Validation on test split

SARD ships with a dedicated test split (570 images, 732 bboxes). We evaluate the best checkpoint on it — this number goes into Bab 2 proposal.

In [ ]:
BEST_PT = Path(PROJECT) / RUN_NAME / 'weights' / 'best.pt'
assert BEST_PT.exists(), f'best weight not found: {BEST_PT}'
print('best checkpoint size (MB):', round(BEST_PT.stat().st_size / 1024**2, 3))

best_model = YOLO(str(BEST_PT))
test_metrics = best_model.val(
    data=str(data_yaml),
    split='test',
    imgsz=1280,
    batch=16,
    plots=True,
    save_json=True,
    name=f'{RUN_NAME}_test',
    project=PROJECT,
)

metrics_summary = {
    'mAP50': float(test_metrics.box.map50),
    'mAP50-95': float(test_metrics.box.map),
    'precision': float(test_metrics.box.mp),
    'recall': float(test_metrics.box.mr),
}
print(json.dumps(metrics_summary, indent=2))

## 5. Export to ONNX for constraint validation

Export the best checkpoint to ONNX FP32 and FP16 variants. These files are what we benchmark on CPU for C-A1 (size) and C-A3 (latency).

In [ ]:
# FP32 ONNX export
onnx_fp32 = best_model.export(
    format='onnx',
    imgsz=1280,
    opset=13,
    dynamic=False,
    simplify=True,
    half=False,
)
print('FP32 ONNX:', onnx_fp32, round(Path(onnx_fp32).stat().st_size / 1024**2, 3), 'MB')

# FP16 ONNX export (halves size, keeps accuracy close to FP32 at CPU inference)
onnx_fp16 = best_model.export(
    format='onnx',
    imgsz=1280,
    opset=13,
    dynamic=False,
    simplify=True,
    half=True,
)
print('FP16 ONNX:', onnx_fp16, round(Path(onnx_fp16).stat().st_size / 1024**2, 3), 'MB')

## 6. Quick GPU-side sanity benchmark

This is **NOT authoritative for C-A3** (C-A3 requires CPU). Use this only to sanity-check that the model works. The real C-A3 validation runs in `00_kaggle_cpu_benchmark.ipynb` on a CPU-only notebook.

In [ ]:
import time, numpy as np
import onnxruntime as ort

sess_fp32 = ort.InferenceSession(onnx_fp32, providers=['CPUExecutionProvider'])
dummy = np.random.randn(1, 3, 1280, 1280).astype(np.float32)

# Warm up
for _ in range(2): sess_fp32.run(None, {sess_fp32.get_inputs()[0].name: dummy})

lats = []
for _ in range(5):
    t = time.perf_counter()
    sess_fp32.run(None, {sess_fp32.get_inputs()[0].name: dummy})
    lats.append(time.perf_counter() - t)

print(f'ONNX FP32 on Kaggle GPU-node CPU (not authoritative!):')
print(f'  min  : {min(lats)*1000:.1f} ms')
print(f'  mean : {sum(lats)/len(lats)*1000:.1f} ms')
print(f'  max  : {max(lats)*1000:.1f} ms')
print('NOTE: GPU notebooks have different CPU SKUs than CPU notebooks.')
print('Authoritative C-A3 validation uses CPU-only Kaggle notebook.')

## 7. Persist artifacts & summary

Save a compact JSON summary alongside the weights so the training run is self-documenting. The proposal Bab 2 + Bab 3 will cite these numbers directly.

In [ ]:
summary = {
    'run_name': RUN_NAME,
    'model': 'YOLOv12s',
    'dataset': 'SARD',
    'imgsz': 1280,
    'epochs_configured': 100,
    'best_checkpoint_mb': round(BEST_PT.stat().st_size / 1024**2, 3),
    'onnx_fp32_mb': round(Path(onnx_fp32).stat().st_size / 1024**2, 3),
    'onnx_fp16_mb': round(Path(onnx_fp16).stat().st_size / 1024**2, 3),
    'test_metrics': metrics_summary,
    'constraint_budget': {
        'c_a1_limit_mb': 50.0,
        'c_a1_this_model_mb': round(Path(onnx_fp16).stat().st_size / 1024**2, 3),
        'c_a1_budget_remaining_mb': round(50.0 - Path(onnx_fp16).stat().st_size / 1024**2, 3),
    },
}
summary_path = WORK / f'{RUN_NAME}_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print()
print(f'Summary saved: {summary_path}')
print()
print('Files to download from /kaggle/working/ before session ends:')
print(f'  - {BEST_PT}')
print(f'  - {onnx_fp32}')
print(f'  - {onnx_fp16}')
print(f'  - {summary_path}')
print(f'  - {Path(PROJECT) / RUN_NAME}/  (full training artifacts incl. plots)')